In [1]:
import pandas as pd
import numpy as np
from glob import glob

In [2]:
filepath_pattern = 'Data/AQI_Pollutant/*.csv'
file_list = sorted(glob(filepath_pattern)) 
dataframes = []
cols_to_keep = ['Date', 'Daily Mean PM2.5 Concentration', 'Daily AQI Value', 'Site Latitude', 'Site Longitude', 'Local Site Name']


In [3]:
for file in file_list:
    df = pd.read_csv(file)
    # df = df.loc[:, df.nunique(dropna=True) > 1]
    # df = df[cols_to_keep]

    non_constant = df.nunique(dropna=True) > 1
    columns_to_keep = non_constant.index[non_constant | non_constant.index.isin(cols_to_keep)]
    df = df[columns_to_keep]
    df['Date'] = pd.to_datetime(df['Date'])
    dataframes.append(df)

merged_df = pd.concat(dataframes, ignore_index=True)
temp = merged_df.groupby(['Site Latitude', 'Site Longitude'])[['Local Site Name']].count()
coords_to_drop = temp[temp['Local Site Name'] == 0].index

merged_df = merged_df[~merged_df.set_index(['Site Latitude', 'Site Longitude']).index.isin(coords_to_drop)]
duplicates = merged_df.duplicated(keep='first')
merged_df = merged_df[~duplicates]
merged_df['Date'] = pd.to_datetime(merged_df['Date'])
merged_df.head();

In [4]:
merged_df = (
    merged_df.sort_values(by='Date')
    .groupby(['Site Latitude', 'Site Longitude', 'Date'], as_index=False)
    .first()
)

merged_df.head() 
#printing this for verification
san_jose_df = merged_df[merged_df['Local Site Name'] == 'San Jose - Jackson']
san_jose_df.head()

,Site Latitude,Site Longitude,Date,Site ID,POC,Daily Mean PM2.5 Concentration,Daily AQI Value,Local Site Name,AQS Parameter Code,AQS Parameter Description,Method Code,Method Description,CBSA Code,CBSA Name,County FIPS Code,County,Daily Obs Count,Percent Complete,Source
295541,37.348497,-121.894898,2014-01-01,60850005,3,60.4,154,San Jose - Jackson,88101,PM2.5 - Local Conditions,170.0,Met One BAM-1020 Mass Monitor w/VSCC,41940.0,"San Jose-Sunnyvale-Santa Clara, CA",85,Santa Clara,NaN,NaN,None
295542,37.348497,-121.894898,2014-01-02,60850005,1,33.0,96,San Jose - Jackson,88101,PM2.5 - Local Conditions,145.0,R & P Model 2025 PM-2.5 Sequential Air Sampler...,41940.0,"San Jose-Sunnyvale-Santa Clara, CA",85,Santa Clara,NaN,NaN,None
295543,37.348497,-121.894898,2014-01-03,60850005,3,32.6,95,San Jose - Jackson,88101,PM2.5 - Local Conditions,170.0,Met One BAM-1020 Mass Monitor w/VSCC,41940.0,"San Jose-Sunnyvale-Santa Clara, CA",85,Santa Clara,NaN,NaN,None
295544,37.348497,-121.894898,2014-01-04,60850005,3,26.4,83,San Jose - Jackson,88101,PM2.5 - Local Conditions,170.0,Met One BAM-1020 Mass Monitor w/VSCC,41940.0,"San Jose-Sunnyvale-Santa Clara, CA",85,Santa Clara,NaN,NaN,None
295545,37.348497,-121.894898,2014-01-05,60850005,5,19.5,70,San Jose - Jackson,88502,Acceptable PM2.5 AQI & Speciation Mass,810.0,Met One SASS/SuperSASS Teflon,41940.0,"San Jose-Sunnyvale-Santa Clara, CA",85,Santa Clara,NaN,NaN,None


In [5]:
grouped = (
    merged_df.groupby(['Site Latitude', 'Site Longitude'])
    .size()
    .reset_index(name='Count')
    .sort_values('Count', ascending=False)
)
top_40_coords = grouped.head(40)[['Site Latitude', 'Site Longitude']]


merged_df_top_40 = merged_df.merge(top_40_coords, on=['Site Latitude', 'Site Longitude'])
merged_df_top_40 = merged_df_top_40.sort_values(by=['Local Site Name', 'Date'], ascending=True).reset_index(drop=True)
merged_df_top_40.head()

,Site Latitude,Site Longitude,Date,Site ID,POC,Daily Mean PM2.5 Concentration,Daily AQI Value,Local Site Name,AQS Parameter Code,AQS Parameter Description,Method Code,Method Description,CBSA Code,CBSA Name,County FIPS Code,County,Daily Obs Count,Percent Complete,Source
0,33.83062,-117.93845,2014-01-01,60590007,3,46.5,128,Anaheim,88502,Acceptable PM2.5 AQI & Speciation Mass,170.0,Met-one BAM-1020 W/PM2.5 SCC,31080.0,"Los Angeles-Long Beach-Anaheim, CA",59,Orange,NaN,NaN,None
1,33.83062,-117.93845,2014-01-02,60590007,1,38.8,109,Anaheim,88101,PM2.5 - Local Conditions,120.0,Andersen RAAS2.5-300 PM2.5 SEQ w/WINS,31080.0,"Los Angeles-Long Beach-Anaheim, CA",59,Orange,NaN,NaN,None
2,33.83062,-117.93845,2014-01-03,60590007,1,29.7,89,Anaheim,88101,PM2.5 - Local Conditions,120.0,Andersen RAAS2.5-300 PM2.5 SEQ w/WINS,31080.0,"Los Angeles-Long Beach-Anaheim, CA",59,Orange,NaN,NaN,None
3,33.83062,-117.93845,2014-01-04,60590007,1,29.9,90,Anaheim,88101,PM2.5 - Local Conditions,120.0,Andersen RAAS2.5-300 PM2.5 SEQ w/WINS,31080.0,"Los Angeles-Long Beach-Anaheim, CA",59,Orange,NaN,NaN,None
4,33.83062,-117.93845,2014-01-05,60590007,11,9.8,52,Anaheim,88502,Acceptable PM2.5 AQI & Speciation Mass,810.0,Met One SASS/SuperSASS Teflon,31080.0,"Los Angeles-Long Beach-Anaheim, CA",59,Orange,NaN,NaN,None


In [6]:
merged_df_top_40 = merged_df_top_40.rename(columns={
    'Site Latitude': 'Latitude',
    'Site Longitude': 'Longitude',
    'Daily Mean PM2.5 Concentration': 'PM2.5',
    'Daily AQI Value': 'AQI',
    'Local Site Name': 'Name'
})

merged_df_top_40.head()

,Latitude,Longitude,Date,Site ID,POC,PM2.5,AQI,Name,AQS Parameter Code,AQS Parameter Description,Method Code,Method Description,CBSA Code,CBSA Name,County FIPS Code,County,Daily Obs Count,Percent Complete,Source
0,33.83062,-117.93845,2014-01-01,60590007,3,46.5,128,Anaheim,88502,Acceptable PM2.5 AQI & Speciation Mass,170.0,Met-one BAM-1020 W/PM2.5 SCC,31080.0,"Los Angeles-Long Beach-Anaheim, CA",59,Orange,NaN,NaN,None
1,33.83062,-117.93845,2014-01-02,60590007,1,38.8,109,Anaheim,88101,PM2.5 - Local Conditions,120.0,Andersen RAAS2.5-300 PM2.5 SEQ w/WINS,31080.0,"Los Angeles-Long Beach-Anaheim, CA",59,Orange,NaN,NaN,None
2,33.83062,-117.93845,2014-01-03,60590007,1,29.7,89,Anaheim,88101,PM2.5 - Local Conditions,120.0,Andersen RAAS2.5-300 PM2.5 SEQ w/WINS,31080.0,"Los Angeles-Long Beach-Anaheim, CA",59,Orange,NaN,NaN,None
3,33.83062,-117.93845,2014-01-04,60590007,1,29.9,90,Anaheim,88101,PM2.5 - Local Conditions,120.0,Andersen RAAS2.5-300 PM2.5 SEQ w/WINS,31080.0,"Los Angeles-Long Beach-Anaheim, CA",59,Orange,NaN,NaN,None
4,33.83062,-117.93845,2014-01-05,60590007,11,9.8,52,Anaheim,88502,Acceptable PM2.5 AQI & Speciation Mass,810.0,Met One SASS/SuperSASS Teflon,31080.0,"Los Angeles-Long Beach-Anaheim, CA",59,Orange,NaN,NaN,None


In [7]:
merged_df_top_40.set_index('Date', inplace=True)
merged_df_top_40.to_csv('Data/top_40_cities_pollution_data.csv', index='Date')

In [8]:
city_locations = merged_df_top_40.groupby(['Latitude', 'Longitude', 'Name']).size().reset_index(name='Count')
city_locations[['Latitude', 'Longitude', 'Name']]

,Latitude,Longitude,Name
0,32.676180,-115.483070,Calexico-Ethel Street
1,33.583018,-117.072202,Temecula
2,33.676490,-117.330980,Lake Elsinore
3,33.830620,-117.938450,Anaheim
4,33.920860,-116.858410,Banning Airport
5,33.996360,-117.492400,Mira Loma (Van Buren)
6,33.999580,-117.416010,Rubidoux
7,34.066590,-118.226880,Los Angeles-North Main Street
8,34.199250,-118.532760,Reseda
9,34.210169,-118.870509,Thousand Oaks
